In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path

from astropy.nddata import CCDData
from astropy.stats import SigmaClip, mad_std
from astropy import units as u
from astropy.table import QTable, vstack
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astroquery.xmatch import XMatch
from astropy.table import Table
from matplotlib.patches import Ellipse

from astroquery.astrometry_net import AstrometryNet

import ccdproc as ccdp

from photutils.aperture import CircularAperture, CircularAnnulus, ApertureStats
from photutils.detection import find_peaks

import sep

# zscale stretch factor
stretch = 4

matplotlib.rcParams.update({'font.size': 12})

%matplotlib inline

### Locate all images

In [ ]:
read_path = '/home/evanmayer/TIM_data/test_events/TIMcam/flight_test/solve/'
img_table_solved = QTable.read(read_path + 'tabulated_solved.fits')

Perform photometry on the best images: daytime, 0.1 s exposure, base gain, solved

In [ ]:
default_gain = abs(img_table_solved['GAINFACT'] - 4.0) < 1e-3
default_exp = abs(img_table_solved['EXPTIME'] - 0.1* u.s) < 1e-3 * u.s
solved = img_table_solved['SOLVED'] > 0
daytime_float = []
for fname in img_table_solved['FILES']:
    if (
        ('10-01_18' in fname)
        or ('10-01_19' in fname)
        or ('10-01_20' in fname)
        or ('10-01_21' in fname)
        or ('10-01_22' in fname)
        or ('10-01_23' in fname)
        or ('10-02_00' in fname)
    ):
        daytime_float.append(True)
        # print(fname)
    else:
        daytime_float.append(False)
daytime_float = np.array(daytime_float)
candidates = (default_gain & default_exp & daytime_float & solved)

In [ ]:
# reduced data path
direc = '/home/evanmayer/TIM_data/test_events/TIMcam/flight_test/cal/'
calibrated_data = Path(direc)
calibrated_data.mkdir(exist_ok=True)

In [ ]:
img_table_solved[candidates]

After an autofocus, one frame is captured with AF settings, and should not be considered. Exclude them on grounds of difference in file size (noise level).

In [ ]:
bkg_mean = img_table_solved[candidates]['FILESIZE']
mu, sig = np.mean(bkg_mean), np.std(bkg_mean)
condition = np.abs(bkg_mean - mu) > 3*sig
outliers = np.where(condition)[0]
inliers = np.where(~condition)[0]
plt.figure()
plt.plot(img_table_solved[candidates]['FILESIZE'])
plt.scatter(np.arange(len(img_table_solved[candidates]))[outliers], img_table_solved[candidates][outliers]['FILESIZE'], color='r')
plt.scatter(np.arange(len(img_table_solved[candidates]))[inliers], img_table_solved[candidates][inliers]['FILESIZE'], color='g')
plt.show()

In [ ]:
good_files = img_table_solved[candidates][inliers]['FILES']
good_basenames = [os.path.basename(f) for f in good_files]

In [ ]:
sci_cal = ccdp.ImageFileCollection('/home/evanmayer/TIM_data/test_events/TIMcam/flight_test/cal/sci/', glob_include=f'*.fits.fz')

### Science: aperture photometry

In [ ]:
def get_photometry(in_table, match_radius=20*u.arcsec, RPmag_limit=9):
    # query_cat = 'vizier:I/355/gaiadr3'
    query_cat = 'vizier:I/259/tyc2'
    results = XMatch.query(
        cat1=in_table,
        colRA1='ra',
        colDec1='dec',
        cat2=query_cat,
        # colRA2='RAJ2000',
        # colDec2='DEJ2000',
        colRA2='RA_ICRS',
        colDec2='DE_ICRS',
        max_distance=match_radius
    )
    # bluer and redder passbands
    # B = results['BPmag']
    # R = results['RPmag']
    B = results['BTmag']
    R = results['VTmag']
    valid = R < RPmag_limit
    return results[valid], B[valid], R[valid]


def sep_flatfield(img):
    z = img.astype(np.float64)
    opts = dict(
        # bw=78,
        # bh=78,
        # fw=16,
        # fh=16
    )
    flat_data = sep.Background(z, **opts)
    return flat_data

In [ ]:
%matplotlib qt
plt.ion()
fig, ax = plt.subplots()
ax.axhline(3, linestyle='--', color='k')

all_results = []
for j, sci in enumerate(sci_cal.files):
    print(sci)
    if os.path.basename(sci) not in good_basenames:
        print('skipping', sci)
        continue
    ccdobj = CCDData.read(calibrated_data / 'sci' / sci)
    img = ccdobj.data
    # img is in e-/s

    border = 100
    mask = np.ones_like(img)
    mask[border:-border, border:-border] = 0 # 1-masked elements are ignored
    mask = mask.astype(bool)

    bkg = sep_flatfield(img)

    data_sub = img - bkg.back()
    m, s = np.mean(data_sub), np.std(data_sub)

    # fig, ax = plt.subplots(figsize=(12,6))
    # ax.imshow(data_sub, vmin=m-0.5*s, vmax=m+0.5*s, origin='lower')

    objects = sep.extract(data_sub, 1.5, err=bkg.globalrms, mask=mask)

    if len(objects) < 1:
        continue

    # do photometry
    flux, fluxerr, flag = sep.sum_circle(data_sub, objects['x'], objects['y'],
                                     3.0, err=bkg.rms(), gain=1.0)

    idx_sort = np.argsort(flux)[::-1]
    flux = flux[idx_sort]
    objects = objects[idx_sort]


    instrumental_mags = -2.5 * np.log10(flux)
    for i in range(len(flux)):
        print("object {:d}: flux = {:f} +/- {:f}, SNR {:f}, mag {:f}".format(i, flux[i], fluxerr[i], flux[i] / fluxerr[i], -2.5 * np.log10(flux[i])))

    # observed coordinates
    wcs = ccdobj.wcs
    radec = wcs.pixel_to_world(objects['x'], objects['y'])
    radec_j2000 = radec.transform_to('fk5')
    tab = Table(
        [radec_j2000.ra, radec_j2000.dec, instrumental_mags, flux, fluxerr],
        names=['ra', 'dec', 'inst_mag', 'flux', 'fluxerr'])
    results, B, R = get_photometry(tab, match_radius=16*u.arcsec)
    # print(results)
    all_results.append(results)

    if j < 2000:
        print(results)
        ax.scatter(R, results['flux'] / results['fluxerr'], alpha=0.05)
        plt.draw()
        plt.pause(0.025)

In [ ]:
plt.show()

In [ ]:
combined_results = vstack(all_results)

In [ ]:
combined_results.write('combined_100ms_gain4.fits', overwrite=True)

In [ ]:
combined_results = Table.read('combined_100ms_gain4.fits')

In [ ]:
results

Check source matching

In [ ]:
%matplotlib qt

fig, ax = plt.subplots(subplot_kw=dict(projection=wcs))
ax.imshow(data_sub, interpolation='nearest', cmap='bone',
                vmin=m-2*s, vmax=m+2*s, origin='lower')
overlay = ax.get_coords_overlay('fk5')
overlay.grid(color='white', ls='dotted')
overlay[0].set_axislabel('Right Ascension (J2000)')
overlay[1].set_axislabel('Declination (J2000)')
ax.scatter(radec_j2000.ra, radec_j2000.dec, s=100, c=flux, transform=ax.get_transform('world'))

ax.scatter(results['ra'], results['dec'], color='k', transform=ax.get_transform('world'))
# ax.scatter(results['RAmdeg'], results['DEmdeg'], color='k', transform=ax.get_transform('world'))
plt.show()

In [ ]:
len(radec), len(results)

In [ ]:
# combined_results['angDist', 'ra', 'dec', 'inst_mag', 'DR3Name', 'BPmag', 'RPmag']
combined_results['angDist', 'ra', 'dec', 'inst_mag', 'BTmag', 'VTmag']

Quick check of linearity and color

In [ ]:
# non_nans = np.where((~np.isnan(combined_results['BPmag'])) & (~np.isnan(combined_results['RPmag'])) & (~np.isnan(combined_results['inst_mag'])))
non_nans = np.where((~np.isnan(combined_results['BTmag'])) & (~np.isnan(combined_results['VTmag'])) & (~np.isnan(combined_results['inst_mag'])))

In [ ]:
fig, ax = plt.subplots()
# ax.scatter(combined_results['inst_mag'][non_nans], combined_results['BPmag'][non_nans])
# ax.scatter(combined_results['inst_mag'][non_nans], combined_results['RPmag'][non_nans])
ax.scatter(combined_results['inst_mag'], combined_results['BTmag'])
ax.scatter(combined_results['inst_mag'], combined_results['VTmag'])
# ax.scatter(results['inst_mag'], results['BPmag'] - results['RPmag'])
ax.invert_xaxis()
ax.invert_yaxis()
ax.set_aspect('equal')
plt.show()

Fit for our red filter zero-point, referred to the Gaia red filter system with a color correction term

In [ ]:
len(non_nans[0]), len(combined_results)

In [ ]:
# 10.1146/annurev.astro.41.082801.100251
from scipy.optimize import curve_fit

def f(x, ZP, a):
    return ZP + a * x

# B0 - B = ZP + a(B - V)
# B0 - B = ZP + aB - aV
# B0 - ZP + aV = aB + B
# (B0 - ZP + aV) / (a + 1) = B

p0 = [0, 0.5]
# xdata = combined_results['BPmag'][non_nans] - combined_results['RPmag'][non_nans]
# ydata = combined_results['inst_mag'][non_nans] - combined_results['RPmag'][non_nans]
xdata = combined_results['BTmag'] - combined_results['VTmag'][non_nans]
ydata = combined_results['inst_mag'][non_nans] - combined_results['VTmag'][non_nans]
popt, pcov = curve_fit(f, xdata, ydata, p0)
ZP_fit, a_fit = popt
print(popt, pcov)

fig, ax = plt.subplots()
ax.scatter(xdata, ydata, alpha=0.05)
ax.plot(xdata, f(xdata, popt[0], popt[1]))
ax.axvline(0, linestyle='--', color='k')
ax.axhline(ZP_fit, linestyle='--', color='k')
ax.invert_xaxis()
ax.invert_yaxis()
ax.set_aspect('equal')
plt.show()

Check results: a good transformation should intersect at y = 0 and have slope = 1

In [ ]:
plt.figure()
# putative_filt = (combined_results['inst_mag'][non_nans] - ZP_fit + a_fit * combined_results['RPmag'][non_nans]) / (a_fit + 1)
# plt.scatter(putative_filt, combined_results['RPmag'][non_nans], alpha=0.05)
putative_filt = (combined_results['inst_mag'][non_nans] - ZP_fit + a_fit * combined_results['VTmag'][non_nans]) / (a_fit + 1)
plt.scatter(putative_filt, combined_results['VTmag'][non_nans], alpha=0.05)
plt.plot(np.arange(0,10), np.arange(0,10), color='C1')
plt.show()

In [ ]:
plt.figure()
plt.hist(putative_filt, bins=10)
plt.show()
len(putative_filt)

In [ ]:
putative_filt.max()

Spot check: convert sky brightness into filter magnitudes/arcsec^2

In [ ]:
# need flipud to show people which way is down because readout of frame lines was inverted
bkg_rate_e_per_s = np.flipud(bkg.back())

fig, ax = plt.subplots()
im = ax.imshow(bkg_rate_e_per_s, origin='lower', interpolation='none')
fig.colorbar(im, ax=ax)
ax.set_title('Background counts: e-/s')

ps_arcsec_per_px = np.mean(proj_plane_pixel_scales(wcs) * 3600) # arcsec/px
print(ps_arcsec_per_px)

# rate per arcsec^2
bkg_rate_per_arcsec2 = bkg_rate_e_per_s / ps_arcsec_per_px**2

fig, ax = plt.subplots()
im = ax.imshow(bkg_rate_per_arcsec2, origin='lower', interpolation='none', cmap='bone')
fig.colorbar(im, ax=ax)
ax.set_title('Background counts: e-/s/(arcsec^2)')

mu = -2.5 * np.log10(bkg_rate_per_arcsec2) - ZP_fit

fig, ax = plt.subplots()
im = ax.imshow(mu, origin='lower', interpolation='none', cmap='viridis_r')
cb = fig.colorbar(im, ax=ax)
cb.ax.invert_yaxis()
ax.set_title('Background counts: mag/arcsec^2')
print(mu.max(), mu.min())

plt.show()

Plot max, min, and median sky mags over time

In [ ]:
ps_arcsec_per_px2 = np.mean(proj_plane_pixel_scales(wcs) * 3600)**2 # arcsec/px

all_mu_maps = []
all_bkg_maps = []
for j, sci in enumerate(sci_cal.files):
    if os.path.basename(sci) not in good_basenames:
        print('skipping', sci)
        continue
    ccdobj = CCDData.read(calibrated_data / 'sci' / sci)
    print(j / len(sci_cal.files), ccdobj.header['EXPTIME'], sci, end='\r')
    img = ccdobj.data
    # img is now in e-/s

    border = 100
    mask = np.ones_like(img)
    mask[border:-border, border:-border] = 0 # 1-masked elements are ignored
    mask = mask.astype(bool)

    bkg = sep_flatfield(img)
    # e- / s in each px
    bkg_rate_e_per_s = bkg.back()
    # rate per arcsec^2
    bkg_rate_per_arcsec2 = bkg_rate_e_per_s / ps_arcsec_per_px2
    # in magnitudes with zero point correction
    mu = -2.5 * np.log10(bkg_rate_per_arcsec2) - ZP_fit
    all_mu_maps.append((mu.max(), np.median(mu), np.mean(mu), mu.min()))
    all_bkg_maps.append((bkg_rate_per_arcsec2.max(), np.median(bkg_rate_per_arcsec2), np.mean(bkg_rate_per_arcsec2), bkg_rate_per_arcsec2.min(), bkg.globalback / ps_arcsec_per_px2))

In [ ]:
ts = img_table_solved[candidates][inliers]['TIMESTAMP'][::50].value

In [ ]:
fig, ax = plt.subplots()
mins = [minn for maxx, medd, meann, minn in all_mu_maps]
meds =[medd for maxx, medd, meann, minn in all_mu_maps]
means =[meann for maxx, medd, meann, minn in all_mu_maps]
maxs = [maxx for maxx, medd, meann, minn in all_mu_maps]
ax.fill_between(ts, maxs, mins, alpha=0.2, color='tomato')
ax.plot(ts, meds, marker='.', color='tomato')
# ax.fill_between(range(len(maxs)), maxs, mins, alpha=0.2, color='tomato')
# ax.plot(range(len(meds)), meds, marker='.', color='tomato')
ax.invert_yaxis()
ax.set_xlabel('Timestamp (UTC)')
ax.set_ylabel('Background (mag/arcsec^2)')
fig.tight_layout()
plt.show()

In [ ]:
maps_reduced = Table((ts, mins, meds, means, maxs), names=['ts', 'mins', 'meds', 'means', 'maxs'])
maps_reduced.write('bkg_summary_mag_arcsec2_100ms_gain4.fits', overwrite=True)

In [ ]:
fig, ax = plt.subplots()
mins = [minn for maxx, medd, meann, minn, gback in all_bkg_maps]
meds =[medd for maxx, medd, meann, minn, gback in all_bkg_maps]
means =[meann for maxx, medd, meann, minn, gback in all_bkg_maps]
maxs = [maxx for maxx, medd, meann, minn, gback in all_bkg_maps]
gbacks = [gback for maxx, medd, meann, minn, gback in all_bkg_maps]
ax.fill_between(ts, maxs, mins, alpha=0.2, color='tomato')
ax.plot(ts, meds, marker='.', color='tomato')
ax.plot(ts, gbacks, marker='.', color='cornflowerblue')
# ax.fill_between(range(len(maxs)), maxs, mins, alpha=0.2, color='tomato')
# ax.plot(range(len(meds)), meds, marker='.', color='tomato')
# ax.invert_yaxis()
ax.set_xlabel('Timestamp (UTC)')
ax.set_ylabel('Background (e-/s/arcsec^2)')
fig.tight_layout()
plt.show()

In [ ]:
maps_reduced = Table((ts, mins, meds, means, maxs, gbacks), names=['ts', 'mins', 'meds', 'means', 'maxs', 'gbacks'])
maps_reduced.write('bkg_summary_e-_s_arcsec2_100ms_gain4.fits', overwrite=True)

In [ ]:
%matplotlib qt

fig, (ax, ax1) = plt.subplots(nrows=2, sharex=True)
exps = [
    'bkg_summary_e-_s_arcsec2_50ms_gain1.fits',
    'bkg_summary_e-_s_arcsec2_100ms_gain1.fits',
    'bkg_summary_e-_s_arcsec2_200ms_gain1.fits',
    'bkg_summary_e-_s_arcsec2_100ms_gain2.fits',
    'bkg_summary_e-_s_arcsec2_100ms_gain4.fits'
]
labels = [
    '50 ms, gain 1',
    '100 ms, gain 1',
    '200 ms, gain 1',
    '100 ms, gain 2',
    '100 ms, gain 4'
]
for i, exp in enumerate(exps):
    maps_reduced = Table.read(exp)
    ts, mins, meds, means, maxs, gbacks = maps_reduced['ts'], maps_reduced['mins'], maps_reduced['meds'], maps_reduced['means'], maps_reduced['maxs'], maps_reduced['gbacks']
    ax.fill_between(ts, maxs, mins, alpha=0.2, color=f'C{i}')
    ax.plot(ts, meds, marker='.', color=f'C{i}', label=labels[i])
    ax1.fill_between(ts, -2.5 * np.log10(maxs), -2.5 * np.log10(mins), alpha=0.2, color=f'C{i}')
    ax1.plot(ts, -2.5 * np.log10(meds), marker='.', color=f'C{i}')
ax.set_ylabel('Background\n(e-/s/arcsec^2)')
plt.figlegend(loc='center right')
ax1.set_xlabel('Timestamp (UTC)')
ax1.invert_yaxis()
ax1.set_ylabel('Instrumental Magnitudes\n(mag/arcsec^2)')

fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
x = img_table_solved['TIMESTAMP'].value
y = img_table_solved['BKG_MEAN'].value / img_table_solved['EXPTIME'].value / img_table_solved['GAINFACT'].value
ax.scatter(x, y)#, c=img_table_solved['CCDTEMP'].value, cmap='viridis')
ax.set_xlabel('t')
ax.set_ylabel('ADU/s')